In [3]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[2]

# Make the src directory importable
sys.path.insert(0, str(project_root))

from src.data_loader import load_cv_data

# Load all five folds
fold_datasets = load_cv_data(project_root)

# Inspect the results
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [4]:

from src.feature_engineering import add_overall_history_features

v12_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    # Generate training features using past-only global prior
    v12_train_df = add_overall_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features using training data for global prior
    v12_val_df = add_overall_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v12_fold_datasets.append(
        (v12_train_df, v12_val_df)
    )

    print(
        f"Fold {fold_number}: "
        f"Training = {v12_train_df.shape}, "
        f"Validation = {v12_val_df.shape}"
    )


Fold 1: Training = (177324, 12), Validation = (42659, 12)
Fold 2: Training = (176273, 12), Validation = (43710, 12)
Fold 3: Training = (170186, 12), Validation = (49797, 12)
Fold 4: Training = (178311, 12), Validation = (41672, 12)
Fold 5: Training = (177838, 12), Validation = (42145, 12)


In [5]:
import numpy as np

fold_train_df, fold_val_df = v12_fold_datasets[0]

# Check the first few values
fold_train_df[
    ["previous_interactions", "log_previous_interactions"]
].head(10)

,previous_interactions,log_previous_interactions
0,0,0.000000
1,1,0.693147
2,2,1.098612
3,3,1.386294
4,4,1.609438
5,5,1.791759
6,6,1.945910
7,7,2.079442
8,8,2.197225
9,9,2.302585


In [6]:
expected = np.log1p(fold_train_df["previous_interactions"])

np.allclose(
    fold_train_df["log_previous_interactions"],
    expected
)

True

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

feature_columns = [
    "overall_smoothed_accuracy",
    "log_previous_interactions"
]

v12_models = []
v12_results = []

for fold_number, (train_df, val_df) in enumerate(v12_fold_datasets, start=1):

    X_train = train_df[feature_columns]
    y_train = train_df["correct"]

    X_val = val_df[feature_columns]
    y_val = val_df["correct"]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    ll = log_loss(y_val, y_pred, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred)

    v12_models.append(model)

    v12_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs
    })

    print(
        f"Fold {fold_number}: "
        f"Log Loss = {ll:.4f}, "
        f"Brier Score = {bs:.4f}"
    )

Fold 1: Log Loss = 0.5944, Brier Score = 0.2035
Fold 2: Log Loss = 0.5900, Brier Score = 0.2013
Fold 3: Log Loss = 0.6042, Brier Score = 0.2080
Fold 4: Log Loss = 0.5947, Brier Score = 0.2036
Fold 5: Log Loss = 0.5860, Brier Score = 0.1997


In [8]:
for fold_number, model in enumerate(v12_models, start=1):
    coefficients = model.named_steps["classifier"].coef_[0]

    print(f"Fold {fold_number}")
    print(f"  Smoothed accuracy coefficient: {coefficients[0]:.4f}")
    print(f"  Log experience coefficient:    {coefficients[1]:.4f}")

Fold 1
  Smoothed accuracy coefficient: 0.6689
  Log experience coefficient:    0.0420
Fold 2
  Smoothed accuracy coefficient: 0.6861
  Log experience coefficient:    0.0379
Fold 3
  Smoothed accuracy coefficient: 0.6715
  Log experience coefficient:    0.0398
Fold 4
  Smoothed accuracy coefficient: 0.6801
  Log experience coefficient:    0.0390
Fold 5
  Smoothed accuracy coefficient: 0.6648
  Log experience coefficient:    0.0440
